# Lab 06 - CSP con DFS, Backtracking y Forward Checking

Autor: Javier Espana
Curso: Inteligencia Artificial 2026

Este notebook implementa y compara dos enfoques de busqueda para CSP:
- DFS con backtracking
- DFS con forward checking (filtering)

Problemas cubiertos:
1. Sudoku de tamano n x n (n cuadrado perfecto >= 4).
2. N-Queens.
3. Extension para obtener todas las soluciones de N-Queens.

In [ ]:
import math
import time
from copy import deepcopy

def print_separator(char='-', width=72):
    print(char * width)

def format_seconds(seconds):
    return f"{seconds:.6f}"

def print_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))

    def format_row(row_values):
        return " | ".join(str(v).ljust(widths[i]) for i, v in enumerate(row_values))

    print(format_row(headers))
    print("-+-".join('-' * w for w in widths))
    for row in rows:
        print(format_row(row))

In [ ]:
class SudokuSolver:
    def __init__(self, n, board):
        root = int(math.isqrt(n))
        if root * root != n or n < 4:
            raise ValueError('n debe ser un cuadrado perfecto >= 4')
        if len(board) != n or any(len(row) != n for row in board):
            raise ValueError('El tablero debe ser de tamano n x n')

        self.n = n
        self.root = root
        self.board = [row[:] for row in board]
        self.nodes_visited = 0

    def reset_stats(self):
        self.nodes_visited = 0

    def _symbol(self, value):
        if value == 0:
            return '.'
        if value <= 9:
            return str(value)
        return chr(ord('A') + value - 10)

    def display(self, board=None):
        board = self.board if board is None else board
        block = self.root
        cell_w = 2 if self.n <= 9 else 3

        for r in range(self.n):
            if r % block == 0 and r != 0:
                print_separator('-', (cell_w * self.n) + (3 * (block - 1)))

            row_parts = []
            for c in range(self.n):
                if c % block == 0 and c != 0:
                    row_parts.append('|')
                row_parts.append(self._symbol(board[r][c]).rjust(cell_w - 1))
            print(' '.join(row_parts))

    def is_valid(self, row, col, num, board=None):
        board = self.board if board is None else board

        for c in range(self.n):
            if board[row][c] == num and c != col:
                return False

        for r in range(self.n):
            if board[r][col] == num and r != row:
                return False

        br = (row // self.root) * self.root
        bc = (col // self.root) * self.root
        for r in range(br, br + self.root):
            for c in range(bc, bc + self.root):
                if board[r][c] == num and (r, c) != (row, col):
                    return False

        return True

    def _find_unassigned_first(self, board):
        for r in range(self.n):
            for c in range(self.n):
                if board[r][c] == 0:
                    return (r, c)
        return None

    def _build_domains(self, board):
        values = set(range(1, self.n + 1))
        domains = {}

        for r in range(self.n):
            for c in range(self.n):
                if board[r][c] != 0:
                    continue

                used = set(board[r][x] for x in range(self.n) if board[r][x] != 0)
                used |= set(board[x][c] for x in range(self.n) if board[x][c] != 0)

                br = (r // self.root) * self.root
                bc = (c // self.root) * self.root
                for rr in range(br, br + self.root):
                    for cc in range(bc, bc + self.root):
                        if board[rr][cc] != 0:
                            used.add(board[rr][cc])

                domains[(r, c)] = values - used
                if not domains[(r, c)]:
                    return None

        return domains

    def _peers(self, row, col):
        peers = set()

        for c in range(self.n):
            if c != col:
                peers.add((row, c))

        for r in range(self.n):
            if r != row:
                peers.add((r, col))

        br = (row // self.root) * self.root
        bc = (col // self.root) * self.root
        for r in range(br, br + self.root):
            for c in range(bc, bc + self.root):
                if (r, c) != (row, col):
                    peers.add((r, c))

        return peers

    def _select_mrv(self, domains):
        return min(domains, key=lambda cell: len(domains[cell]))

    def solve_backtracking(self):
        self.reset_stats()
        board = self.board

        def dfs():
            cell = self._find_unassigned_first(board)
            if cell is None:
                return True

            r, c = cell
            self.nodes_visited += 1

            for num in range(1, self.n + 1):
                if self.is_valid(r, c, num, board):
                    board[r][c] = num
                    if dfs():
                        return True
                    board[r][c] = 0

            return False

        solved = dfs()
        return solved, [row[:] for row in board]

    def solve_filtering(self):
        self.reset_stats()
        board = self.board

        domains = self._build_domains(board)
        if domains is None:
            return False, [row[:] for row in board]

        def dfs(domains):
            if not domains:
                return True

            cell = self._select_mrv(domains)
            r, c = cell
            self.nodes_visited += 1

            for num in sorted(domains[cell]):
                board[r][c] = num

                new_domains = {k: set(v) for k, v in domains.items() if k != cell}
                failed = False

                for peer in self._peers(r, c):
                    if peer in new_domains and num in new_domains[peer]:
                        new_domains[peer].remove(num)
                        if not new_domains[peer]:
                            failed = True
                            break

                if not failed and dfs(new_domains):
                    return True

                board[r][c] = 0

            return False

        solved = dfs(domains)
        return solved, [row[:] for row in board]

In [ ]:
class NQueensSolver:
    def __init__(self, n):
        if n < 1:
            raise ValueError('n debe ser >= 1')
        self.n = n
        self.board = [-1] * n
        self.nodes_visited = 0

    def reset_stats(self):
        self.board = [-1] * self.n
        self.nodes_visited = 0

    def is_valid(self, row, col, board=None):
        board = self.board if board is None else board
        for r in range(row):
            c = board[r]
            if c == col:
                return False
            if abs(c - col) == abs(r - row):
                return False
        return True

    def display(self, board=None):
        board = self.board if board is None else board
        for r in range(self.n):
            line = ['.'] * self.n
            if board[r] != -1:
                line[board[r]] = 'Q'
            print(' '.join(line))

    def solve_backtracking(self):
        self.reset_stats()
        board = self.board

        def dfs(row):
            if row == self.n:
                return True

            self.nodes_visited += 1

            for col in range(self.n):
                if self.is_valid(row, col, board):
                    board[row] = col
                    if dfs(row + 1):
                        return True
                    board[row] = -1

            return False

        solved = dfs(0)
        return solved, board[:]

    def solve_filtering(self):
        self.reset_stats()
        board = self.board
        domains = {row: set(range(self.n)) for row in range(self.n)}

        def dfs(row, domains):
            if row == self.n:
                return True

            self.nodes_visited += 1

            for col in sorted(domains[row]):
                board[row] = col
                new_domains = {r: set(v) for r, v in domains.items()}
                failed = False

                for rr in range(row + 1, self.n):
                    if col in new_domains[rr]:
                        new_domains[rr].remove(col)

                    d = rr - row
                    c1 = col - d
                    c2 = col + d

                    if c1 in new_domains[rr]:
                        new_domains[rr].remove(c1)
                    if c2 in new_domains[rr]:
                        new_domains[rr].remove(c2)

                    if not new_domains[rr]:
                        failed = True
                        break

                if not failed and dfs(row + 1, new_domains):
                    return True

                board[row] = -1

            return False

        solved = dfs(0, domains)
        return solved, board[:]

    def solve_all_backtracking(self):
        self.reset_stats()
        board = self.board
        solutions = []

        def dfs(row):
            if row == self.n:
                solutions.append(board[:])
                return

            self.nodes_visited += 1

            for col in range(self.n):
                if self.is_valid(row, col, board):
                    board[row] = col
                    dfs(row + 1)
                    board[row] = -1

        dfs(0)
        return solutions

## Parte 1 - Sudoku

Se resuelven 4 escenarios solicitados:
- 4x4
- 9x9 facil
- 9x9 extremo
- 16x16

In [ ]:
# Casos Sudoku
sudoku_4x4 = [
    [1, 0, 0, 4],
    [0, 4, 1, 0],
    [0, 1, 4, 0],
    [4, 0, 0, 1],
]

sudoku_9x9_facil = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

sudoku_9x9_extremo = [
    [8, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 3, 6, 0, 0, 0, 0, 0],
    [0, 7, 0, 0, 9, 0, 2, 0, 0],
    [0, 5, 0, 0, 0, 7, 0, 0, 0],
    [0, 0, 0, 0, 4, 5, 7, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 3, 0],
    [0, 0, 1, 0, 0, 0, 0, 6, 8],
    [0, 0, 8, 5, 0, 0, 0, 1, 0],
    [0, 9, 0, 0, 0, 0, 4, 0, 0],
]

# Construimos un 16x16 valido y ocultamos celdas para formar el problema
solucion_16x16 = [[((r * 4 + r // 4 + c) % 16) + 1 for c in range(16)] for r in range(16)]
sudoku_16x16 = deepcopy(solucion_16x16)
for r in range(16):
    for c in range(16):
        if (r + c) % 3 == 0:
            sudoku_16x16[r][c] = 0

sudoku_cases = [
    ('Sudoku 4x4', sudoku_4x4),
    ('Sudoku 9x9 facil', sudoku_9x9_facil),
    ('Sudoku 9x9 extremo', sudoku_9x9_extremo),
    ('Sudoku 16x16', sudoku_16x16),
]

In [ ]:
def run_sudoku_benchmarks(cases):
    rows = []
    solutions = {}

    for case_name, board in cases:
        for method in ('backtracking', 'filtering'):
            solver = SudokuSolver(len(board), board)
            start = time.perf_counter()

            if method == 'backtracking':
                solved, solution = solver.solve_backtracking()
            else:
                solved, solution = solver.solve_filtering()

            elapsed = time.perf_counter() - start
            rows.append([
                case_name,
                method,
                solved,
                solver.nodes_visited,
                format_seconds(elapsed),
            ])
            solutions[(case_name, method)] = solution

    return rows, solutions

sudoku_rows, sudoku_solutions = run_sudoku_benchmarks(sudoku_cases)
print('Resultados Sudoku (nodos visitados y tiempo en segundos)')
print_table(sudoku_rows, headers=['Caso', 'Metodo', 'Resuelto', 'Nodos', 'Tiempo (s)'])

print('\nMuestra de solucion por metodo y caso:')
for case_name, board in sudoku_cases:
    print_separator('=')
    print(case_name)
    for method in ('backtracking', 'filtering'):
        print_separator()
        print(f'Metodo: {method}')
        SudokuSolver(len(board), board).display(sudoku_solutions[(case_name, method)])

## Parte 2 - N Queens

Se evalua el problema para:
- N = 4
- N = 12
- N = 8
- N = 15

In [ ]:
nqueen_cases = [4, 12, 8, 15]

def run_nqueens_benchmarks(cases):
    rows = []
    solutions = {}

    for n in cases:
        for method in ('backtracking', 'filtering'):
            solver = NQueensSolver(n)
            start = time.perf_counter()

            if method == 'backtracking':
                solved, solution = solver.solve_backtracking()
            else:
                solved, solution = solver.solve_filtering()

            elapsed = time.perf_counter() - start
            rows.append([
                f'N={n}',
                method,
                solved,
                solver.nodes_visited,
                format_seconds(elapsed),
            ])
            solutions[(n, method)] = solution

    return rows, solutions

nq_rows, nq_solutions = run_nqueens_benchmarks(nqueen_cases)
print('Resultados N-Queens (nodos visitados y tiempo en segundos)')
print_table(nq_rows, headers=['Caso', 'Metodo', 'Resuelto', 'Nodos', 'Tiempo (s)'])

print('\nSoluciones encontradas (representacion por filas):')
for n in nqueen_cases:
    print_separator('=')
    print(f'N={n}')
    for method in ('backtracking', 'filtering'):
        print(f'  {method}: {nq_solutions[(n, method)]}')

## Parte 3 - Todas las soluciones de N-Queens

Se modifica la estrategia de backtracking para enumerar todas las soluciones para:
- N = 4
- N = 5
- N = 6

In [ ]:
all_cases = [4, 5, 6]
all_rows = []

for n in all_cases:
    solver = NQueensSolver(n)
    start = time.perf_counter()
    solutions = solver.solve_all_backtracking()
    elapsed = time.perf_counter() - start

    all_rows.append([
        f'N={n}',
        len(solutions),
        solver.nodes_visited,
        format_seconds(elapsed),
    ])

print('Conteo de todas las soluciones en N-Queens')
print_table(all_rows, headers=['Caso', 'Total soluciones', 'Nodos', 'Tiempo (s)'])

## Discusion tecnica

Con base en las corridas de este notebook, forward checking suele visitar menos nodos que backtracking puro.

Razon computacional:
1. Backtracking detecta conflicto tarde, cuando ya hizo asignaciones profundas.
2. Forward checking propaga restricciones al momento de asignar y poda temprano ramas inviables.
3. Menor tamano efectivo del arbol de busqueda implica menos nodos y menor tiempo, sobre todo en instancias mas restrictivas.

En los experimentos de Sudoku 9x9 (especialmente el caso extremo) y N-Queens grandes, el efecto de poda es visible en nodos y tiempo.

Nota: en algunas instancias pequenas o con muchas pistas, las diferencias de tiempo pueden ser pequenas por el overhead de mantener dominios.